# Session 2: Cypher — Da Basi a Query Reali


Benvenuti alla Sessione 2! In questo notebook approfondiremo Cypher partendo dal dataset Org Graph caricato nella Sessione 1.

**Cosa faremo:**
1. Connettersi a Neo4j e ricaricare il dataset
2. Pattern MATCH avanzati (path variables, multi-hop)
3. MERGE vs CREATE: idempotenza
4. Filtri WHERE (AND, OR, IN, CONTAINS)
5. Aggregazioni e subquery
6. Date, liste, map
7. Query tuning con PROFILE / EXPLAIN
8. APOC essentials
9. Esercizi

**Dataset:** Org Graph (238 persone, 93 progetti, 55 skill)


## 1. Setup e Connessione


Prima di tutto installiamo le dipendenze necessarie e connettiamoci al database.
Il driver Python per Neo4j (v6+) supporta la sintassi `execute_query` che useremo durante la sessione.


In [ ]:
%pip install neo4j python-dotenv pandas -q


In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv("../../.env")
URI  = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PASS = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(URI, auth=(USER, PASS))
driver.verify_connectivity()
print("Connesso a Neo4j!")


## 2. Recap: il Dataset Org Graph


Ricarichiamo il dataset nel database (se non lo avete già fatto nella Sessione 1, eseguite le celle qui sotto).
Se il dataset è già presente, le operazioni MERGE non creeranno duplicati.


In [ ]:
df = pd.read_csv("../../data/org_graph.csv")
print(f"Righe: {len(df)}, Colonne: {list(df.columns)}")
df.head(3)


In [ ]:
core_cols = ["employee_id", "employee_name", "location", "role", "supervisor_name"]
skill_cols = [c for c in df.columns if c not in core_cols]
print(f"Skill columns ({len(skill_cols)}): {skill_cols}")


## 3. Pattern MATCH Avanzati


### Path variables e multi-hop

In Cypher possiamo assegnare pattern complessi a variabili e navigare su più hop con sintassi `[*min..max]`:
- `-[:REL*1..4]->` da 1 a 4 hop
- `-[:REL*]->` profondità illimitata
- `shortestPath()` per il cammino minimo


In [ ]:
# Path a lunghezza variabile — catena di comando
query = '''
MATCH path = (p:Person)-[:REPORTS_TO*1..4]->(m:Person)
RETURN p.employee_name AS impiegato,
       m.employee_name AS manager,
       length(path) AS livelli
ORDER BY livelli
LIMIT 20
'''
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r['impiegato']} → {r['manager']} ({r['livelli']} livelli)")


In [ ]:
# ShortestPath — cammino minimo attraverso progetti condivisi
query = '''
MATCH (a:Person {employee_id: 1}), (b:Person {employee_id: 100})
MATCH path = shortestPath((a)-[:WORKED_ON*]-(b))
RETURN [n IN nodes(path) | n.employee_name] AS percorso,
       length(path) AS hop
'''
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"Percorso ({r['hop']} hop): {' → '.join(r['percorso'])}")


## 4. MERGE vs CREATE: Idempotenza


- **CREATE**: inserisce sempre — se il nodo esiste già con un constraint UNIQUE, fallisce.
- **MERGE**: cerca il pattern; se non esiste, lo crea (match-or-create).
- Idempotente: eseguire N volte produce lo stesso risultato.


In [ ]:
# MERGE è sicuro — può essere eseguito più volte
query = '''
MERGE (p:Person {employee_id: 999})
  SET p.employee_name = "Test User", p.location = "Test"
RETURN p.employee_name, p.location
'''
records, _, _ = driver.execute_query(query)
print("MERGE eseguito:", records[0])
# Seconda esecuzione — stesso risultato
records, _, _ = driver.execute_query(query)
print("MERGE idempotente:", records[0])
# Pulizia
driver.execute_query('MATCH (p:Person {employee_id: 999}) DETACH DELETE p')
print("Pulito")


In [ ]:
# MERGE su relazione — cerca o crea
query = '''
MATCH (p:Person {employee_id: 1}), (s:Skill {nome: 'Python'})
MERGE (p)-[h:HAS_SKILL]->(s)
  SET h.livello = "Senior"
RETURN p.employee_name, s.nome, h.livello
'''
records, _, _ = driver.execute_query(query)
print(f"{records[0]['p.employee_name']} ha skill {records[0]['s.nome']} a livello {records[0]['h.livello']}")


## 5. Filtri WHERE Avanzati


Cypher supporta operatori logici e funzioni per filtrare:

- `AND`, `OR`, `NOT`
- `IN [...]` per membership
- `CONTAINS`, `STARTS WITH`, `ENDS WITH` per pattern testuali
- `IS NULL`, `IS NOT NULL`


In [ ]:
# Combinazioni AND/OR con IN
query = '''
MATCH (p:Person)
WHERE (p.location IN ["Roma", "Milano"])
  AND p.employee_name STARTS WITH "M"
RETURN p.employee_name, p.location
ORDER BY p.employee_name
LIMIT 10
'''
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r['p.employee_name']:20s} | {r['p.location']}")


In [ ]:
# Ricerca testuale CONTAINS
query = '''
MATCH (s:Skill)
WHERE s.nome CONTAINS "cloud"
RETURN s.nome AS skill
ORDER BY skill
'''
records, _, _ = driver.execute_query(query)
print(f"Skill con 'cloud': {len(records)}")
for r in records:
    print(f"  - {r['skill']}")


## 6. Aggregazioni


Le funzioni di aggregazione in Cypher lavorano senza GROUP BY esplicito:
- `count(*)` conta righe, `count(expr)` conta non-null
- `collect(expr)` raccoglie valori in una lista
- `avg()`, `sum()`, `min()`, `max()`


In [ ]:
# Skill più diffuse
query = '''
MATCH (p:Person)-[:HAS_SKILL]->(s:Skill)
RETURN s.nome AS skill, count(*) AS persone
ORDER BY persone DESC
LIMIT 10
'''
records, _, _ = driver.execute_query(query)
print(f"{'Skill':25s} | Persone")
print("-" * 35)
for r in records:
    print(f"{r['skill']:25s} | {r['persone']}")


In [ ]:
# Livello medio per skill
query = '''
MATCH (p:Person)-[h:HAS_SKILL]->(s:Skill)
RETURN s.nome AS skill,
       round(avg(h.livello), 1) AS media,
       count(*) AS persone
ORDER BY media DESC
LIMIT 10
'''
records, _, _ = driver.execute_query(query)
print(f"{'Skill':25s} | Media | Persone")
print("-" * 40)
for r in records:
    print(f"{r['skill']:25s} | {r['media']:5.1f} | {r['persone']}")


In [ ]:
# COLLECT — progetti per persona
query = '''
MATCH (p:Person)-[:WORKED_ON]->(proj:Project)
RETURN p.employee_name AS persona,
       collect(proj.nome) AS progetti,
       count(proj) AS num_progetti
ORDER BY num_progetti DESC
LIMIT 10
'''
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r['persona']:20s} | {r['num_progetti']:2d} progetti: {', '.join(r['progetti'])}")


## 7. Ordinamento e Paginazione


- `ORDER BY ... ASC/DESC` — anche su multiple colonne
- `LIMIT n` — prime n righe
- `SKIP n` — salta n righe (paginazione)


In [ ]:
# Paginazione: seconda pagina (record 11-20)
query = '''
MATCH (p:Person)
RETURN p.employee_name AS nome, p.location AS location
ORDER BY nome
SKIP 10 LIMIT 10
'''
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r['nome']:20s} | {r['location']}")


## 8. Subquery con CALL { ... }


Le subquery permettono di annidare query correlate o fare UNION tra risultati.
Sintassi: `CALL { subquery }` con `UNION` per combinare.


In [ ]:
# UNION: skill e progetti in una lista unica
query = '''
CALL { MATCH (s:Skill) RETURN s.nome AS nome, 'Skill' AS tipo }
UNION
CALL { MATCH (p:Project) RETURN p.nome AS nome, 'Progetto' AS tipo }
RETURN nome, tipo
ORDER BY nome
LIMIT 20
'''
records, _, _ = driver.execute_query(query)
for r in records:
    print(f"{r['nome']:30s} | {r['tipo']}")


In [ ]:
# Conteggi multipli per persona con subquery correlate
query = '''
MATCH (p:Person)
RETURN p.employee_name AS nome,
       CALL { MATCH (p)-[:HAS_SKILL]->(s:Skill) RETURN count(*) } AS num_skill,
       CALL { MATCH (p)-[:WORKED_ON]->(proj:Project) RETURN count(*) } AS num_progetti
ORDER BY num_skill DESC
LIMIT 10
'''
records, _, _ = driver.execute_query(query)
print(f"{'Nome':20s} | Skill | Progetti")
print("-" * 40)
for r in records:
    print(f"{r['nome']:20s} | {r['num_skill']:5d} | {r['num_progetti']:8d}")


## 9. Date, Liste e Mappe


Cypher supporta date, operazioni su liste e mappe:
- `date()`, `datetime()`, `date("2026-07-01")`
- Liste: `range(0, n)`, list comprehension, `UNWIND`
- Mappe: `{key: value}`, accesso con `.key`


In [ ]:
# Date
query = "RETURN date() AS oggi, date('2026-07-15') AS prima_sessione"
records, _, _ = driver.execute_query(query)
print(f"Oggi: {records[0]['oggi']}")
print(f"Prima sessione: {records[0]['prima_sessione']}")


In [ ]:
# Range e UNWIND per generare righe
query = '''
UNWIND range(2020, 2026) AS anno
RETURN anno
'''
records, _, _ = driver.execute_query(query)
for r in records:
    print(r['anno'])


In [ ]:
# List comprehension
query = '''
WITH [1, 2, 3, 4, 5] AS numeri
RETURN [x IN numeri WHERE x > 2 | x * 10] AS trasformati
'''
records, _, _ = driver.execute_query(query)
print(f"Trasformati: {records[0]['trasformati']}")


## 10. Query Tuning: PROFILE ed EXPLAIN


- **EXPLAIN**: mostra il piano di esecuzione stimato (non esegue)
- **PROFILE**: esegue la query e mostra statistiche reali (dbHits, rows)

Cosa cercare nel piano:
- `NodeIndexSeek` / `NodeUniqueIndexSeek`: usa un indice
- `AllNodesScan`: scandisce l'intero database (cattivo)
- `CartesianProduct`: pattern non collegati (da evitare)


In [ ]:
# EXPLAIN — piano stimato
query = "EXPLAIN MATCH (p:Person)-[:HAS_SKILL]->(s:Skill) WHERE p.location = 'Roma' RETURN p.employee_name, s.nome"
records, summary, _ = driver.execute_query(query)
print("=== PIANO STIMATO ===")
for op in summary.plan.operators:
    print(f'  {op}')


In [ ]:
# PROFILE — esecuzione con metriche
query = "PROFILE MATCH (p:Person {location: 'Roma'})-[:HAS_SKILL]->(s:Skill) RETURN p.employee_name, s.nome"
records, summary, _ = driver.execute_query(query)
print("=== RISULTATI ===")
for r in records:
    print(f"{r['p.employee_name']:20s} | {r['s.nome']}")


## 11. APOC: Procedure Standard


APOC (Awesome Procedures on Cypher) è la libreria standard di procedure per Neo4j.
Su AuraDB molte procedure APOC sono già disponibili. Qui vediamo alcune delle più utili.


In [ ]:
# Verifica APOC disponibile
try:
    records, _, _ = driver.execute_query('RETURN apoc.version() AS versione')
    print(f"APOC versione: {records[0]['versione']}")
except Exception as e:
    print(f"APOC non disponibile su questa istanza: {e}")


In [ ]:
# Unire due liste con apoc.coll.zip
query = "RETURN apoc.coll.zip([1, 2, 3], ['a', 'b', 'c']) AS zippato"
records, _, _ = driver.execute_query(query)
print(records[0]['zippato'])


In [ ]:
# Pulizia testo
query = "RETURN apoc.text.clean('Cypher Query Language!') AS pulito"
records, _, _ = driver.execute_query(query)
print(f"Pulito: {records[0]['pulito']}")


## 12. Esercizi


Ora provate voi! Scrivete le query Cypher per risolvere i seguenti problemi sul dataset Org Graph.

> Suggerimento: usate `driver.execute_query(query)` come visto sopra, oppure sperimentate direttamente nel Neo4j Browser.


### Esercizio 1 — Skill più diffuse


Trovate le 5 skill più diffuse tra i dipendenti (conteggio persone).


In [ ]:
# Il vostro codice qui



### Esercizio 2 — Progetti per persona


Per ogni persona, contate il numero di progetti a cui ha partecipato. Mostrate solo le prime 10 persone.


In [ ]:
# Il vostro codice qui



### Esercizio 3 — Cammino minimo


Trovate il cammino minimo tra due impiegati a vostra scelta (usate `shortestPath`).


In [ ]:
# Il vostro codice qui



### Esercizio 4 — Skill overlap


Trovate coppie di impiegati che condividono almeno 3 skill.


In [ ]:
# Il vostro codice qui



### Esercizio 5 — Subquery


Usando subquery correlate, per ogni persona mostrate: nome, numero di skill, numero di progetti, e nome del supervisor diretto.


In [ ]:
# Il vostro codice qui



### Esercizio 6 — PROFILE


Prendete una query complessa e analizzatela con PROFILE. Cercate `AllNodesScan` o `CartesianProduct` e provate a ottimizzarla.


In [ ]:
# Il vostro codice qui



## Riepilogo

Oggi abbiamo visto:

| Concetto | Query di esempio |
|----------|-----------------|
| Path a lunghezza variabile | `(p)-[:REPORTS_TO*1..4]->(m)` |
| ShortestPath | `shortestPath((a)-[:WORKED_ON*]-(b))` |
| MERGE idempotente | `MERGE (p:Person {id: $id}) SET ...` |
| Filtri WHERE | `WHERE location IN [...] AND nome CONTAINS ...` |
| Aggregazioni | `count(*)`, `collect()`, `avg()` |
| Subquery | `CALL { MATCH ... }` |
| PROFILE/EXPLAIN | Diagnostica performance |
| APOC | `apoc.coll.*`, `apoc.text.*` |

**Prossima sessione: Data Modeling & Import** — progetteremo modelli a grafo robusti, useremo LOAD CSV e refattorizzeremo Org Graph.


In [ ]:
# Chiudi connessione
driver.close()
print("Connessione chiusa")
